# Part 26a: sklearn Pipeline -- Foundations

**Dataset:** Hotel Bookings (117,000 reservations, 32 features)

**Covers sections 1-5:** project setup, Pipeline, ColumnTransformer, leakage prevention, model family comparison.


Part 25 gave you the sklearn API contract: every transformer fits on training data and applies to any split. You applied that contract step by step -- scale, then fit, then evaluate. The problem is that 'step by step' breaks down in cross-validation: if you scale before the fold split, the validation fold contaminates the scaler's parameters. This chapter solves that with `Pipeline`, which guarantees every transformer re-fits inside each fold. Part 26b adds hyperparameter search and a final model card.

## 1. Setup

All imports at the top in PEP 8 order: stdlib, then third-party in alphabetical package order, then local.

In [ ]:
from __future__ import annotations

# stdlib
from pathlib import Path
from typing import Any

# data
import numpy as np

# hyperparameter search
import optuna
import pandas as pd
from scipy.stats import loguniform, uniform

# sklearn -- transformers and composition
from sklearn.compose import ColumnTransformer

# sklearn -- models
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# sklearn -- metrics
from sklearn.metrics import (
    classification_report,
    f1_score,
)

# sklearn -- model selection
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

# persistence
import shutil

# tables
from great_tables import GT, md as gt_md
import joblib

# visualisation
from lets_plot import (
    LetsPlot,
    aes,
    as_discrete,
    geom_bar,
    geom_line,
    geom_point,
    ggplot,
    labs,
    scale_color_manual,
    scale_fill_manual,
)

# logging
from loguru import logger

# project branding
from ark.plot.gt_style import metrics_report, themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, BRAND_PALETTE, DANGER, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html()

### 1.1 Project constants

In [ ]:
DATA_URL: str = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
DATA_PATH: Path = Path("data/hotel_bookings.csv")
MODEL_DIR: Path = Path("models")
RANDOM_STATE: int = 42

### 1.2 Load the hotel bookings dataset

Same load function as Part 25. If you ran nb04 first, the data is cached on disk.

In [ ]:
def load_hotel_data(
    path: Path = DATA_PATH,
    url: str = DATA_URL,
) -> pd.DataFrame:
    """Load hotel bookings, downloading from url on first call."""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        raw = pd.read_csv(url)
        raw.to_csv(path, index=False)
        logger.success(f"Downloaded {len(raw):,} rows to {path}")
    else:
        raw = pd.read_csv(path)
        logger.info(f"Loaded {len(raw):,} rows from cache: {path}")
    df = raw[(raw["adr"] > 0) & (raw["adr"] <= 1000)].reset_index(drop=True)
    logger.info(f"After cleaning: {len(df):,} rows")
    return df

### 1.3 Execute the loader

In [ ]:
df: pd.DataFrame = load_hotel_data()
logger.info(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

### 1.4 Define the full feature set

Part 25 used six numerical features. Here we add six categorical features, which requires `ColumnTransformer`.

In [ ]:
NUMERICAL_FEATURES: list[str] = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "previous_cancellations",
    "booking_changes",
    "total_of_special_requests",
    "days_in_waiting_list",
]
CATEGORICAL_FEATURES: list[str] = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "customer_type",
    "deposit_type",
]
ALL_FEATURES: list[str] = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGET_CLF: str = "is_canceled"

### 1.5 Preview the categorical features

Knowing the cardinality of each categorical column tells us how many binary columns `OneHotEncoder` will produce.

In [ ]:
cat_summary: list[dict[str, Any]] = [
    {"feature": c, "n_unique": df[c].nunique(), "values": str(sorted(df[c].unique().tolist())[:6])}
    for c in CATEGORICAL_FEATURES
]
cat_df: pd.DataFrame = pd.DataFrame(cat_summary)
(
    themed_gt(GT(cat_df), n_rows=len(cat_df)).tab_header(
        title=gt_md("**Categorical Features**"),
        subtitle="Cardinality determines the number of one-hot columns",
    )
)

### 1.6 Train/test split

**Critical rule:** split before any preprocessing. The test set must never influence transformer parameters.

In [ ]:
X: pd.DataFrame = df[ALL_FEATURES].copy()
y: np.ndarray = df[TARGET_CLF].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
logger.info(f"Train: {X_train.shape}  Test: {X_test.shape}")
logger.info(f"Train cancel rate: {y_train.mean():.2%}  Test cancel rate: {y_test.mean():.2%}")

## 2. The sklearn Pipeline

**From Part 25 (nb04 Section 7.3):** we wrapped scaler + model inside a `Pipeline` to prevent leakage in cross-validation. Now we formalise this pattern.

<div style="font-family:monospace;background:#F8FAFC;border:1px solid #E2E8F0;border-radius:8px;padding:14px 18px;margin:12px 0;font-size:13px;line-height:1.9">
raw X &rarr; <b>StandardScaler</b> &rarr; X_scaled &rarr; <b>LogisticRegression</b> &rarr; predictions<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(Transformer)&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(Predictor)<br><br>
<code>pipe.fit(X_train, y_train)</code> &rarr; fits BOTH stages on training data only<br>
<code>pipe.predict(X_test)</code>&nbsp;&nbsp;&nbsp;&nbsp; &rarr; transforms then predicts in one call
</div>

A `Pipeline` is a Predictor: it accepts `fit(X, y)` and `predict(X)`. Internally it chains every stage in order.

### 2.1 Define a simple Pipeline (numerical features only)

We replicate the Part 25 setup -- numerical features only -- to isolate the Pipeline concept before adding `ColumnTransformer`.

In [ ]:
pipe_simple: Pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

### 2.2 Fit the simple Pipeline on numerical features only

In [ ]:
X_train_num: pd.DataFrame = X_train[NUMERICAL_FEATURES]
X_test_num: pd.DataFrame = X_test[NUMERICAL_FEATURES]
pipe_simple.fit(X_train_num, y_train)
logger.success("Simple pipeline fitted")

### 2.3 Inspect the fitted stages

`named_steps` gives dictionary access to each stage. `scaler.mean_` confirms the scaler learned from training data only.

In [ ]:
fitted_scaler: StandardScaler = pipe_simple.named_steps["scaler"]
logger.info(f"Scaler means (first 4): {fitted_scaler.mean_[:4].round(2)}")
logger.info(f"Pipeline steps: {list(pipe_simple.named_steps.keys())}")

### 2.4 Evaluate the simple Pipeline on the test set

In [ ]:
y_pred_simple: np.ndarray = pipe_simple.predict(X_test_num)
f1_simple: float = f1_score(y_test, y_pred_simple)
logger.info(f"Simple pipeline F1 (numerical only): {f1_simple:.3f}")

::: {.callout-note icon=false}
## <i class="bi bi-info-circle-fill" style="color:#0369A1"></i>&nbsp; Pipeline as a Predictor

Once built, a `Pipeline` has `fit`, `predict`, and `score` -- same as any sklearn model. You can pass it directly to `cross_val_score`, `GridSearchCV`, or `RandomizedSearchCV`.

The key guarantee: during cross-validation, each transformer is **re-fitted** on the fold's training split, never on the validation split.
:::

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 1: Pipeline with RobustScaler

Replace `StandardScaler` with `RobustScaler` in `pipe_simple` and compare F1 scores.

```python
from sklearn.preprocessing import RobustScaler
# Your code here
```

**Question:** does it improve on the imbalanced dataset?
:::

## 3. ColumnTransformer

**From Part 25 (Section 2):** Transformers only handle numerical matrices by default. Categorical columns need `OneHotEncoder`. `ColumnTransformer` routes different column subsets to different transformers and concatenates the results.

<div style="font-family:monospace;background:#F8FAFC;border:1px solid #E2E8F0;border-radius:8px;padding:14px 18px;margin:12px 0;font-size:13px;line-height:1.9">
Input DataFrame<br>
|-- num: [lead_time, stays_*, adults, ...]  &rarr; SimpleImputer &rarr; StandardScaler &rarr; 8 columns<br>
|-- cat: [hotel, meal, market_segment, ...]  &rarr; SimpleImputer &rarr; OneHotEncoder  &rarr; ~30 columns<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&rarr; concatenated numpy array
</div>

Real-world data also has missing values. We add `SimpleImputer` to each sub-pipeline to handle nulls *before* scaling or encoding.

### 3.1 Define the numerical transformer sub-pipeline

`SimpleImputer(strategy="median")` fills missing values with the column median before scaling. On this clean hotel dataset there are no missing values -- but defensive imputation is good practice for production pipelines.

In [ ]:
numeric_transformer: Pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

### 3.2 Define the categorical transformer sub-pipeline

`SimpleImputer(strategy="most_frequent")` fills missing categories before encoding. `handle_unknown="ignore"` silently converts unseen categories to all-zero rows at inference time -- safer than raising an error in production.

In [ ]:
categorical_transformer: Pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)

### 3.3 Assemble the ColumnTransformer

Each tuple is `(name, transformer, columns)`. Columns can be a list of names (our case) or a dtype selector.

In [ ]:
preprocessor: ColumnTransformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERICAL_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

### 3.4 Fit the ColumnTransformer on training data

`fit_transform` fits and transforms in one call. We do this manually here to inspect the output; in the full pipeline we just call `pipe.fit(...)`.

In [ ]:
X_prep: np.ndarray = preprocessor.fit_transform(X_train)
logger.info(f"Input shape:  {X_train.shape}")
logger.info(f"Output shape: {X_prep.shape}")
logger.info(f"Numerical columns: {len(NUMERICAL_FEATURES)}")
logger.info(f"Categorical columns after OHE: {X_prep.shape[1] - len(NUMERICAL_FEATURES)}")

### 3.5 Inspect generated feature names

`get_feature_names_out()` returns the full list of column names after transformation -- useful for feature importance analysis.

In [ ]:
feature_names: list[str] = preprocessor.get_feature_names_out().tolist()
logger.info(f"Total features after preprocessing: {len(feature_names)}")
logger.info(f"First 8 names: {feature_names[:8]}")
logger.info(f"Last 8 names:  {feature_names[-8:]}")

### 3.6 Build the full Pipeline: ColumnTransformer + LogisticRegression

In [ ]:
pipe_full: Pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

### 3.7 Fit the full Pipeline on ALL features

In [ ]:
pipe_full.fit(X_train, y_train)
logger.success("Full pipeline fitted on all features")

### 3.8 Compare simple pipeline vs full pipeline

In [ ]:
y_pred_full: np.ndarray = pipe_full.predict(X_test)
f1_full: float = f1_score(y_test, y_pred_full)
logger.info(f"Simple pipeline (6 numerical features) F1: {f1_simple:.3f}")
logger.info(f"Full pipeline  (8 num + 6 cat features) F1: {f1_full:.3f}")

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 2: add arrival_date_month

Add `arrival_date_month` to `CATEGORICAL_FEATURES` and re-run the full pipeline.

```python
# Your code here
```

**Question:** does adding the booking month improve F1?
:::

## 4. How Pipeline Prevents Leakage in Cross-Validation

**From Part 24 (ML Workflow, Section 5):** the training set used for CV must have NO information from the validation fold. Manual preprocessing before `cross_val_score` breaks this guarantee.

<div style="font-family:monospace;background:#FEF2F2;border:1px solid #FECACA;border-radius:8px;padding:14px 18px;margin:12px 0;font-size:13px;line-height:1.9">
<b style="color:#DC2626">Wrong (leaky cross-validation):</b><br>
&nbsp;&nbsp;preprocessor.fit_transform(X_all_train) &rarr; X_preprocessed (scaler saw all folds)<br>
&nbsp;&nbsp;cross_val_score(model_only, X_preprocessed, y) &rarr; optimistically low error<br><br>
<b style="color:#059669">Correct (Pipeline inside cross_val_score):</b><br>
&nbsp;&nbsp;cross_val_score(Pipeline([preprocessor, model]), X_raw, y)<br>
&nbsp;&nbsp;&nbsp;&rarr; fold 1: scaler.fit(folds 2-5), scaler.transform(fold 1)<br>
&nbsp;&nbsp;&nbsp;&rarr; fold 2: scaler.fit(folds 1,3-5), scaler.transform(fold 2) ...
</div>

### 4.1 The wrong approach (for illustration only)

Do NOT run this cell on real data -- it is shown to illustrate what leakage looks like.

In [ ]:
# WRONG: preprocessor sees ALL training rows before any fold is held out.
# The scaler's mean_ includes validation-fold rows, contaminating CV estimates.
#
# X_prep_wrong = preprocessor.fit_transform(X_train)
# cross_val_score(
#     LogisticRegression(class_weight="balanced", max_iter=1000),
#     X_prep_wrong, y_train, cv=StratifiedKFold(5), scoring="f1",
# )

logger.warning("The above is intentionally NOT run -- it demonstrates the leaky pattern.")

### 4.2 The correct approach: pass the Pipeline to cross_val_score

Passing the full Pipeline to `cross_val_score` guarantees the preprocessor is re-fitted inside each fold.

In [ ]:
skf: StratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores: np.ndarray = cross_val_score(
    pipe_full,
    X_train,
    y_train,
    cv=skf,
    scoring="f1",
    n_jobs=-1,
)
logger.info(f"CV F1 per fold: {cv_scores.round(3)}")
logger.info(f"Mean +/- Std: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

::: {.callout-warning icon=false}
## <i class="bi bi-bug-fill" style="color:#DC2626"></i>&nbsp; Common Mistake: fit_transform before cross_val_score

```python
# LEAKY:
X_scaled = preprocessor.fit_transform(X_train)
cross_val_score(model, X_scaled, y_train, cv=StratifiedKFold(5))

# CORRECT:
pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
cross_val_score(pipe, X_train, y_train, cv=StratifiedKFold(5))
```

The leaky version uses a scaler fitted on ALL training rows -- including the validation fold -- producing an optimistically low error that will not hold in production.
:::

## 5. Comparing Model Families

**From Part 23 (What Is Machine Learning?, Section 2):** different model families make different assumptions. Before tuning any single model, compare a few families with default hyperparameters using cross-validation to find the most promising starting point.

We evaluate three families on the same pipeline: LogisticRegression (linear), RandomForest (bagging ensemble), and GradientBoosting (boosting ensemble).

### 5.1 Define the candidate model families

In [ ]:
MODEL_CANDIDATES: dict[str, Any] = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=50,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=50,
        random_state=RANDOM_STATE,
    ),
}

### 5.2 Cross-validate each candidate with the same preprocessor

We reuse the same `preprocessor` for all candidates -- only the final estimator changes. This ensures a fair comparison.

In [ ]:
comparison_results: list[dict[str, Any]] = []
for name, model in MODEL_CANDIDATES.items():
    candidate_pipe = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    scores: np.ndarray = cross_val_score(
        candidate_pipe,
        X_train,
        y_train,
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        scoring="f1",
    )
    comparison_results.append(
        {
            "Model": name,
            "F1_mean": float(scores.mean()),
            "F1_std": float(scores.std()),
        }
    )
    logger.info(f"{name:<22} F1 = {scores.mean():.3f} +/- {scores.std():.3f}")

### 5.3 Display the comparison table

In [ ]:
comparison_df: pd.DataFrame = pd.DataFrame(comparison_results)
metrics_report(
    comparison_df,
    metrics=["F1_mean", "F1_std"],
    maximize_cols=["F1_mean"],
    minimize_cols=["F1_std"],
    title="Model Family Comparison (default hyperparameters)",
    subtitle="5-fold stratified CV on training set | Hotel Booking Demand",
)

::: {.callout-note icon=false}
## <i class="bi bi-info-circle-fill" style="color:#0369A1"></i>&nbsp; Model comparison before tuning

Always compare a few model families with **default** hyperparameters before tuning any single one. The best default is a strong signal that the model's inductive bias suits your data. Tuning a model that already performs poorly with defaults rarely produces competitive results -- invest tuning budget in the most promising family.
:::

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'><span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> What's Next</span><br><br>Part 26b continues with hyperparameter search (GridSearchCV, RandomizedSearchCV, Optuna), feature importance analysis, and final evaluation with a model card.</div>